# `fairlearn-fhe` quickstart

Encrypted fairness audit on the UCI Adult dataset, end-to-end:

1. Plaintext baseline (Fairlearn).
2. **Mode A** — encrypted predictions, plaintext sensitive features.
3. **Mode B** — encrypted predictions and encrypted per-row group masks.
4. Audit envelope + signed envelope + CLI verification.

The encrypted verdicts agree with the plaintext baseline within CKKS noise tolerance (default `< 1e-3` abs).

In [ ]:
# %pip install --upgrade 'fairlearn-fhe[signing]' fairlearn scikit-learn pandas

## Train a baseline classifier

In [ ]:
from fairlearn.datasets import fetch_adult
from fairlearn.metrics import (
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = fetch_adult(as_frame=True)
X_raw = data.data
y_true = (data.target == ">50K").astype(int)
sensitive = X_raw["sex"].astype(str)

X = X_raw.drop(columns=["sex"])
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y_true, sensitive, test_size=0.3, random_state=12345, stratify=y_true
)
numeric = X_train.select_dtypes(include="number").columns
categorical = X_train.select_dtypes(exclude="number").columns
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)
model = Pipeline(
    [("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))]
).fit(X_train, y_train)
y_pred = model.predict(X_test)
len(y_pred), float(y_pred.mean())

## Plaintext baseline

In [ ]:
plain = {
    "demographic_parity_difference": demographic_parity_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
    "equalized_odds_difference": equalized_odds_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
    "equal_opportunity_difference": equal_opportunity_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
}
plain

## Mode A — encrypted `y_pred`, plaintext sensitive features

The auditor still holds plaintext `y_test` and `sf_test`; only the model's predictions cross the trust boundary as ciphertext. Depth 1 per metric.

In [ ]:
from fairlearn_fhe import build_context, encrypt
from fairlearn_fhe.metrics import (
    demographic_parity_difference as enc_dp,
)
from fairlearn_fhe.metrics import (
    equal_opportunity_difference as enc_eopp,
)
from fairlearn_fhe.metrics import (
    equalized_odds_difference as enc_eo,
)

ctx = build_context(backend="tenseal")
y_pred_enc = encrypt(ctx, y_pred.astype(float))

mode_a = {
    "demographic_parity_difference": enc_dp(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
    "equalized_odds_difference": enc_eo(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
    "equal_opportunity_difference": enc_eopp(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
}
for k in plain:
    delta = abs(plain[k] - mode_a[k])
    print(f"{k}: plain={plain[k]:.6f} fhe={mode_a[k]:.6f} |Δ|={delta:.2e}")

## Mode B — encrypted `y_pred` *and* encrypted per-row group masks

The auditor sees only the K plaintext per-group rates (decrypted at the audit boundary). Per-row group membership stays encrypted. Depth 2 per metric — the extra ct×ct multiply is the price.

In [ ]:
from fairlearn_fhe import encrypt_sensitive_features

sf_enc = encrypt_sensitive_features(ctx, sf_test, y_true=y_test)
mode_b = {
    "demographic_parity_difference": enc_dp(
        y_test, y_pred_enc, sensitive_features=sf_enc
    ),
    "equalized_odds_difference": enc_eo(
        y_test, y_pred_enc, sensitive_features=sf_enc
    ),
    "equal_opportunity_difference": enc_eopp(
        y_test, y_pred_enc, sensitive_features=sf_enc
    ),
}
for k in plain:
    delta = abs(plain[k] - mode_b[k])
    print(f"{k}: plain={plain[k]:.6f} fhe(B)={mode_b[k]:.6f} |Δ|={delta:.2e}")

## Audit envelope

`audit_metric` packages the verdict into a tamper-evident `MetricEnvelope` capturing parameter-set hash, observed depth, op counts, input hashes, and a UTC timestamp. The envelope is JSON-serialisable and validates against the published [JSON Schema](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/api/envelope.md).

In [ ]:
from fairlearn_fhe import audit_metric

env = audit_metric(
    "demographic_parity_difference",
    y_true=y_test,
    y_pred=y_pred_enc,
    sensitive_features=sf_test,
    ctx=ctx,
    min_group_size=30,
)
env_dict = env.to_dict()
{k: env_dict[k] for k in [
    "metric_name", "value", "observed_depth", "op_counts", "n_samples", "n_groups", "trust_model"
]}

## Sign the envelope

Optional Ed25519 signing for regulator handoff. Requires `pip install 'fairlearn-fhe[signing]'`.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives.serialization import (
    Encoding,
    NoEncryption,
    PrivateFormat,
    PublicFormat,
)

from fairlearn_fhe import sign_envelope, verify_envelope_signature

sk = Ed25519PrivateKey.generate()
sk_pem = sk.private_bytes(Encoding.PEM, PrivateFormat.PKCS8, NoEncryption())
pk_pem = sk.public_key().public_bytes(Encoding.PEM, PublicFormat.SubjectPublicKeyInfo)

signed = sign_envelope(env_dict, sk_pem)
errors = verify_envelope_signature(signed, pk_pem)
print("signed:", signed["signature"]["algorithm"])
print("signature errors:", errors)

## CLI verification (no FHE backend required)

An auditor or regulator can verify the envelope without importing TenSEAL or OpenFHE — `fairlearn-fhe verify` only needs the JSON and (optionally) the PEM public key.

In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "envelope.json"
    p.write_text(json.dumps(signed))
    pk_path = Path(td) / "pk.pem"
    pk_path.write_bytes(pk_pem)
    # Prefer the installed `fairlearn-fhe` script; fall back to invoking
    # the module directly so this cell works in fresh environments where
    # the entry point is not yet on PATH.
    cmd_head = (
        ["fairlearn-fhe"] if shutil.which("fairlearn-fhe")
        else [sys.executable, "-m", "fairlearn_fhe.cli"]
    )
    out = subprocess.run(
        cmd_head + [
            "verify", str(p),
            "--public-key", str(pk_path),
            "--max-age", "86400",
            "--min-security-bits", "128",
            "--json",
        ],
        capture_output=True, text=True,
    )
    print(out.stdout)


## Trust models reference

| Mode | Encrypted | Plaintext | Depth | Use when |
| --- | --- | --- | --- | --- |
| A (default) | `y_pred` | `y_true`, `sensitive_features`, group counts | 1 | Routine internal audit; auditor is allowed to know per-row group membership. |
| B | `y_pred`, per-row group masks | `y_true`, group counts | 2 | Regulator-facing audit; auditor cannot be trusted with per-row sensitive features. |

See [`docs/threat-model.md`](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/threat-model.md) for the formal trust-boundary specification.

## Next steps

- Browse the [supported metrics catalogue](https://bader82t.github.io/fairlearn-fhe/metrics/) (35 functions in v0.2.x).
- Run the included [benchmarks](https://github.com/BAder82t/fairlearn-fhe/blob/main/benchmarks/bench_metrics.py) on your own data.
- Read [`docs/threat-model.md`](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/threat-model.md) before deploying Mode B.